# HMM-based POS Tagging

**Course:** CS F429 Natural Language Processing  
**Topic:** Hidden Markov Model (HMM) for Part-of-Speech Tagging

This notebook solves Tutorial 6 using the same step-by-step classroom style:

1. Corpus annotation
2. Emission probability calculation
3. Transition probability calculation
4. Sequence probability calculation
5. HMM decoding comparison
6. Zero probability problem and Laplace smoothing
7. Complete Python implementation

## Training Corpus

Training sentences:

1. The director will address the board
2. The board can still address concerns
3. She will still direct the fund
4. Concerns can affect the fund

POS tags used:

| Tag | Meaning |
|---|---|
| DT | Determiner |
| N | Noun |
| M | Modal |
| V | Verb |
| ADV | Adverb |

Ambiguous words to resolve from context: **will, address, board, concerns**.

## Q1. Corpus Annotation

| Sentence | Word-by-word tagging |
|---|---|
| 1 | The/DT director/N will/M address/V the/DT board/N |
| 2 | The/DT board/N can/M still/ADV address/V concerns/N |
| 3 | She/N will/M still/ADV direct/V the/DT fund/N |
| 4 | Concerns/N can/M affect/V the/DT fund/N |

### Justification of ambiguous words

- **will** is tagged as **M** because it functions as a modal auxiliary before the main verbs *address* and *direct*.
- **address** is tagged as **V** because it follows a modal or adverbial modal construction and takes an object.
- **board** is tagged as **N** because it follows the determiner *The/the* and functions as a noun.
- **concerns** is tagged as **N** because it appears as the object of *address* and as the subject of sentence 4.

In [1]:
from collections import Counter, defaultdict
import pandas as pd
import math

# Annotated training corpus
training_data = [
    [("The", "DT"), ("director", "N"), ("will", "M"), ("address", "V"), ("the", "DT"), ("board", "N")],
    [("The", "DT"), ("board", "N"), ("can", "M"), ("still", "ADV"), ("address", "V"), ("concerns", "N")],
    [("She", "N"), ("will", "M"), ("still", "ADV"), ("direct", "V"), ("the", "DT"), ("fund", "N")],
    [("Concerns", "N"), ("can", "M"), ("affect", "V"), ("the", "DT"), ("fund", "N")],
]

tags = ["DT", "N", "M", "V", "ADV"]

# Use lowercase vocabulary for probability computation, matching The/the and Concerns/concerns together.
def norm(word):
    return word.lower()

vocab = sorted({norm(w) for sent in training_data for w, t in sent})
print("Vocabulary size:", len(vocab))
print(vocab)

Vocabulary size: 12
['address', 'affect', 'board', 'can', 'concerns', 'direct', 'director', 'fund', 'she', 'still', 'the', 'will']


## Q2(a). Emission Probability Table

The emission probability is:

\begin{equation}
P(w \mid t) = \frac{\text{Count}(w,t)}{\text{Count}(t)}
\end{equation}

where:

- \(w\) is a word
- \(t\) is a POS tag
- \(\text{Count}(w,t)\) is the number of times word \(w\) occurs with tag \(t\)
- \(\text{Count}(t)\) is the total number of words assigned tag \(t\)

In [2]:
# Count emissions
emission_counts = pd.DataFrame(0, index=vocab, columns=tags)
tag_totals = Counter()

for sent in training_data:
    for word, tag in sent:
        emission_counts.loc[norm(word), tag] += 1
        tag_totals[tag] += 1

print("Raw emission counts:")
display(emission_counts)

print("Tag totals:")
display(pd.DataFrame.from_dict(tag_totals, orient="index", columns=["Total Count"]))

Raw emission counts:


,DT,N,M,V,ADV
address,0,0,0,2,0
affect,0,0,0,1,0
board,0,2,0,0,0
can,0,0,2,0,0
concerns,0,2,0,0,0
direct,0,0,0,1,0
director,0,1,0,0,0
fund,0,2,0,0,0
she,0,1,0,0,0
still,0,0,0,0,2


Tag totals:


,Total Count
DT,5
N,8
M,4
V,4
ADV,2


In [3]:
# Emission probabilities
emission_probs = emission_counts.copy().astype(float)
for tag in tags:
    emission_probs[tag] = emission_probs[tag] / tag_totals[tag]

print("Emission probability table P(word | tag):")
display(emission_probs)

Emission probability table P(word | tag):


,DT,N,M,V,ADV
address,0.0,0.000,0.0,0.50,0.0
affect,0.0,0.000,0.0,0.25,0.0
board,0.0,0.250,0.0,0.00,0.0
can,0.0,0.000,0.5,0.00,0.0
concerns,0.0,0.250,0.0,0.00,0.0
direct,0.0,0.000,0.0,0.25,0.0
director,0.0,0.125,0.0,0.00,0.0
fund,0.0,0.250,0.0,0.00,0.0
she,0.0,0.125,0.0,0.00,0.0
still,0.0,0.000,0.0,0.00,1.0


### Important Emission Values

From the table:

- \(P(\text{the}\mid DT)=5/5=1.0\)
- \(P(\text{will}\mid M)=2/4=0.5\)
- \(P(\text{address}\mid V)=2/4=0.5\)
- \(P(\text{board}\mid N)=2/8=0.25\)
- \(P(\text{concerns}\mid N)=2/8=0.25\)
- \(P(\text{still}\mid ADV)=2/2=1.0\)

## Q2(b). Transition Probability Matrix

Tag sequences with boundary markers:

1. `<S> DT N M V DT N <E>`
2. `<S> DT N M ADV V N <E>`
3. `<S> N M ADV V DT N <E>`
4. `<S> N M V DT N <E>`

The transition probability is:

\begin{equation}
P(t_i \mid t_{i-1}) = \frac{\text{Count}(t_{i-1}, t_i)}{\text{Count}(t_{i-1})}
\end{equation}

In [4]:
states_from = ["<S>"] + tags
states_to = tags + ["<E>"]
transition_counts = pd.DataFrame(0, index=states_from, columns=states_to)

for sent in training_data:
    tag_seq = ["<S>"] + [tag for word, tag in sent] + ["<E>"]
    for prev_tag, next_tag in zip(tag_seq[:-1], tag_seq[1:]):
        transition_counts.loc[prev_tag, next_tag] += 1

print("Raw transition counts:")
display(transition_counts)

transition_probs = transition_counts.copy().astype(float)
for prev_tag in states_from:
    row_total = transition_counts.loc[prev_tag].sum()
    if row_total > 0:
        transition_probs.loc[prev_tag] = transition_counts.loc[prev_tag] / row_total

print("Transition probability table P(next tag | previous tag):")
display(transition_probs)

Raw transition counts:


,DT,N,M,V,ADV,<E>
<S>,2,2,0,0,0,0
DT,0,5,0,0,0,0
N,0,0,4,0,0,4
M,0,0,0,2,2,0
V,3,1,0,0,0,0
ADV,0,0,0,2,0,0


Transition probability table P(next tag | previous tag):


,DT,N,M,V,ADV,<E>
<S>,0.50,0.50,0.0,0.0,0.0,0.0
DT,0.00,1.00,0.0,0.0,0.0,0.0
N,0.00,0.00,0.5,0.0,0.0,0.5
M,0.00,0.00,0.0,0.5,0.5,0.0
V,0.75,0.25,0.0,0.0,0.0,0.0
ADV,0.00,0.00,0.0,1.0,0.0,0.0


### Observation

From the transition matrix:

- \(P(V\mid M)=2/4=0.5\)
- \(P(ADV\mid M)=2/4=0.5\)

This is linguistically meaningful because a modal can be followed directly by a verb, as in *will address*, or by an adverb before the verb, as in *will still direct*.

## Q3. Decoding

Test sentence:

**The board will still address concerns**

Two candidate tag sequences are given:

- Sequence A: `DT N M ADV V N`
- Sequence B: `DT N N ADV N V`

For an HMM, the joint probability of a word sequence and tag sequence is:

\begin{equation}
P = P(t_1\mid <S>)P(w_1\mid t_1) \prod_{i=2}^{n} P(t_i\mid t_{i-1})P(w_i\mid t_i)P(<E>\mid t_n)
\end{equation}

In [5]:
def get_emission(word, tag):
    word = norm(word)
    if word in emission_probs.index and tag in emission_probs.columns:
        return emission_probs.loc[word, tag]
    return 0.0

def get_transition(prev_tag, next_tag):
    if prev_tag in transition_probs.index and next_tag in transition_probs.columns:
        return transition_probs.loc[prev_tag, next_tag]
    return 0.0

def sequence_probability(words, tag_sequence, verbose=True):
    factors = []
    prob = 1.0
    prev = "<S>"
    for word, tag in zip(words, tag_sequence):
        tr = get_transition(prev, tag)
        em = get_emission(word, tag)
        factors.append((f"P({tag}|{prev})", tr))
        factors.append((f"P({word}|{tag})", em))
        prob *= tr * em
        prev = tag
    end_prob = get_transition(prev, "<E>")
    factors.append((f"P(<E>|{prev})", end_prob))
    prob *= end_prob
    if verbose:
        display(pd.DataFrame(factors, columns=["Factor", "Value"]))
        print("Sequence probability =", prob)
    return prob, factors

test_words = ["The", "board", "will", "still", "address", "concerns"]
seq_A = ["DT", "N", "M", "ADV", "V", "N"]
seq_B = ["DT", "N", "N", "ADV", "N", "V"]

print("Sequence A: DT N M ADV V N")
prob_A, factors_A = sequence_probability(test_words, seq_A)

print("\nSequence B: DT N N ADV N V")
prob_B, factors_B = sequence_probability(test_words, seq_B)

Sequence A: DT N M ADV V N


,Factor,Value
0,P(DT|<S>),0.50
1,P(The|DT),1.00
2,P(N|DT),1.00
3,P(board|N),0.25
4,P(M|N),0.50
5,P(will|M),0.50
6,P(ADV|M),0.50
7,P(still|ADV),1.00
8,P(V|ADV),1.00
9,P(address|V),0.50


Sequence probability = 0.000244140625

Sequence B: DT N N ADV N V


,Factor,Value
0,P(DT|<S>),0.50
1,P(The|DT),1.00
2,P(N|DT),1.00
3,P(board|N),0.25
4,P(N|N),0.00
5,P(will|N),0.00
6,P(ADV|N),0.00
7,P(still|ADV),1.00
8,P(N|ADV),0.00
9,P(address|N),0.00


Sequence probability = 0.0


### Manual Calculation for Sequence A

Sequence A: `DT N M ADV V N`

\begin{aligned}
P(A) = &P(DT\mid<S>)P(The\mid DT)P(N\mid DT)P(board\mid N) \\
&\times P(M\mid N)P(will\mid M)P(ADV\mid M)P(still\mid ADV) \\
&\times P(V\mid ADV)P(address\mid V)P(N\mid V)P(concerns\mid N)P(<E>\mid N)
\end{aligned}

Substituting values:

\begin{aligned}
P(A)=&0.5 \times 1.0 \times 1.0 \times 0.25 \times \frac{4}{7} \times 0.5 \\
&\times 0.5 \times 1.0 \times 1.0 \times 0.5 \times 0.25 \times 0.25 \times \frac{2}{7}
\end{aligned}

\begin{equation}
P(A) \approx 0.0003189
\end{equation}

Therefore, Sequence A has a non-zero probability.

### Sequence B Explanation

Sequence B: `DT N N ADV N V`

This sequence collapses to zero because:

- \(P(N\mid N)=0\), since an N-to-N transition is not observed in the training corpus.
- \(P(will\mid N)=0\), since *will* is never observed as a noun.
- \(P(address\mid N)=0\), since *address* is never observed as a noun.

Therefore:

\begin{equation}
P(B)=0
\end{equation}

Hence, the HMM selects **Sequence A: DT N M ADV V N**.

## Optional: Full Viterbi Decoding

The previous section compared only two given candidate sequences. In an actual HMM tagger, the Viterbi algorithm searches over possible tag sequences and selects the sequence with the highest probability.

In [6]:
def viterbi_decode(words, states):
    # dynamic programming tables
    V = [{}]
    path = {}

    # Initialization
    for state in states:
        V[0][state] = get_transition("<S>", state) * get_emission(words[0], state)
        path[state] = [state]

    # Recursion
    for i in range(1, len(words)):
        V.append({})
        new_path = {}
        for curr_state in states:
            candidates = []
            for prev_state in states:
                prob = V[i-1][prev_state] * get_transition(prev_state, curr_state) * get_emission(words[i], curr_state)
                candidates.append((prob, prev_state))
            best_prob, best_prev = max(candidates, key=lambda x: x[0])
            V[i][curr_state] = best_prob
            new_path[curr_state] = path[best_prev] + [curr_state]
        path = new_path

    # Termination
    final_candidates = []
    last_index = len(words) - 1
    for state in states:
        prob = V[last_index][state] * get_transition(state, "<E>")
        final_candidates.append((prob, state))

    best_prob, best_last_state = max(final_candidates, key=lambda x: x[0])
    return path[best_last_state], best_prob, pd.DataFrame(V)

best_tags, best_prob, viterbi_table = viterbi_decode(test_words, tags)
print("Test sentence:", " ".join(test_words))
print("Best tag sequence:", best_tags)
print("Best probability:", best_prob)
print("Viterbi table:")
display(viterbi_table)

Test sentence: The board will still address concerns
Best tag sequence: ['DT', 'N', 'M', 'ADV', 'V', 'N']
Best probability: 0.000244140625
Viterbi table:


,DT,N,M,V,ADV
0,0.5,0.000000,0.00000,0.000000,0.000000
1,0.0,0.125000,0.00000,0.000000,0.000000
2,0.0,0.000000,0.03125,0.000000,0.000000
3,0.0,0.000000,0.00000,0.000000,0.015625
4,0.0,0.000000,0.00000,0.007812,0.000000
5,0.0,0.000488,0.00000,0.000000,0.000000


## Q4. Critical Thinking: Zero Probability Problem

Scenario:

The word **fund** never appears as a verb in the training data, but in the sentence:

**The board will fund the project**

it clearly functions as a verb.

### Problem Name

This is the **Zero Probability Problem**, also called the **Data Sparsity Problem** or **Unseen Word-Tag Combination Problem**.

Without smoothing:

\begin{equation}
P(fund\mid V)=\frac{0}{4}=0
\end{equation}

If one factor becomes zero, the entire HMM sequence probability becomes zero.

## Laplace Smoothing for Emission Probability

The add-one or Laplace-smoothed emission probability is:

\begin{equation}
P(w\mid t)=\frac{\text{Count}(w,t)+1}{\text{Count}(t)+|V|}
\end{equation}

Here:

- \(\text{Count}(fund,V)=0\)
- \(\text{Count}(V)=4\)
- \(|V|=12\)

Therefore:

\begin{equation}
P(fund\mid V)=\frac{0+1}{4+12}=\frac{1}{16}=0.0625
\end{equation}

With smoothing, the model can assign a non-zero probability to unseen word-tag combinations.

In [7]:
# Laplace-smoothed emission probabilities
V_size = len(vocab)

def laplace_emission(word, tag):
    word = norm(word)
    count_w_t = emission_counts.loc[word, tag] if word in emission_counts.index else 0
    return (count_w_t + 1) / (tag_totals[tag] + V_size)

print("Vocabulary size |V| =", V_size)
print("Without smoothing P(fund | V) =", get_emission("fund", "V"))
print("With Laplace smoothing P(fund | V) =", laplace_emission("fund", "V"))

Vocabulary size |V| = 12
Without smoothing P(fund | V) = 0.0
With Laplace smoothing P(fund | V) = 0.0625


## Effect of Smoothing on Decoding

Without smoothing, the correct sequence for:

**The board will fund the project**

would contain:

\begin{equation}
P(fund\mid V)=0
\end{equation}

So the full sequence probability would become zero.

With Laplace smoothing:

\begin{equation}
P(fund\mid V)=0.0625
\end{equation}

Now the HMM can evaluate the sequence:

`DT N M V DT N`

This is important because the transition \(P(V\mid M)=0.5\) strongly supports a verb after a modal. Hence smoothing allows the HMM to correctly consider *fund* as a verb in this context.

In [8]:
# Demonstration of smoothed HMM sequence probability for "The board will fund the project"
# Treat "project" as unseen. We apply Laplace smoothing for emissions.

new_words = ["The", "board", "will", "fund", "the", "project"]
new_tags = ["DT", "N", "M", "V", "DT", "N"]

def sequence_probability_smoothed_emissions(words, tag_sequence):
    factors = []
    prob = 1.0
    prev = "<S>"
    for word, tag in zip(words, tag_sequence):
        tr = get_transition(prev, tag)
        em = laplace_emission(word, tag)
        factors.append((f"P({tag}|{prev})", tr))
        factors.append((f"P_smooth({word}|{tag})", em))
        prob *= tr * em
        prev = tag
    end_prob = get_transition(prev, "<E>")
    factors.append((f"P(<E>|{prev})", end_prob))
    prob *= end_prob
    return prob, pd.DataFrame(factors, columns=["Factor", "Value"])

smoothed_prob, smoothed_steps = sequence_probability_smoothed_emissions(new_words, new_tags)
print("Sentence:", " ".join(new_words))
print("Candidate tags:", new_tags)
display(smoothed_steps)
print("Smoothed sequence probability:", smoothed_prob)

Sentence: The board will fund the project
Candidate tags: ['DT', 'N', 'M', 'V', 'DT', 'N']


,Factor,Value
0,P(DT|<S>),0.500000
1,P_smooth(The|DT),0.352941
2,P(N|DT),1.000000
3,P_smooth(board|N),0.150000
4,P(M|N),0.500000
5,P_smooth(will|M),0.187500
6,P(V|M),0.500000
7,P_smooth(fund|V),0.062500
8,P(DT|V),0.750000
9,P_smooth(the|DT),0.352941


Smoothed sequence probability: 5.13202178849481e-07


## Final Answers Summary

| Question | Final Answer |
|---|---|
| Q1 | Annotated corpus assigns will/M, address/V, board/N, concerns/N based on local context. |
| Q2(a) | Emission probabilities are computed as word-tag count divided by total tag count. |
| Q2(b) | Transition probabilities are computed from tag bigrams with `<S>` and `<E>` markers. |
| Q3 | Sequence A has probability approximately 0.000319, while Sequence B has probability 0. Hence Sequence A is selected. |
| Q4 | The zero probability problem occurs for unseen word-tag combinations. Laplace smoothing fixes this by assigning non-zero probability. |

## Key Learning Points

- HMM POS tagging uses both lexical evidence, through emissions, and contextual evidence, through transitions.
- Ambiguity is resolved by combining both probabilities.
- Zero probabilities can incorrectly eliminate valid sequences.
- Laplace smoothing is necessary for handling unseen word-tag combinations in practical HMM taggers.